# RAG 09: Embeddings and Chroma Indexing

This notebook finishes the offline RAG pipeline.

It loads the final chunks, embeds them with Ollama, and stores them in Chroma. The chat script will read this index later.

In [ ]:
from pathlib import Path
import json
import os
import shutil

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

load_dotenv()

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent


def repo_path(value: str) -> Path:
    path = Path(value)
    if path.is_absolute():
        return path
    return REPO_ROOT / path


CHUNKS_PATH = repo_path(os.getenv("OWASP_LLM_CHUNKS_PATH", "data/owasp_top10_llm_chunks.jsonl"))
CHROMA_PERSIST_DIR = repo_path(os.getenv("CHROMA_PERSIST_DIR", "data/chroma_owasp_llm"))
COLLECTION_NAME = "owasp_top10_llm"

if not CHUNKS_PATH.exists():
    raise FileNotFoundError(
        f"Missing {CHUNKS_PATH}. Run notebooks/08_rag_chunking_strategies.ipynb first."
    )

## Load Final Chunks

Each row has a stable chunk id, text content, and metadata for citations.

In [ ]:
chunk_documents = []
chunk_ids = []

with CHUNKS_PATH.open(encoding="utf-8") as file:
    for line in file:
        row = json.loads(line)
        chunk_documents.append(
            Document(page_content=row["content"], metadata=row["metadata"])
        )
        chunk_ids.append(row["id"])

print({"chunks_loaded": len(chunk_documents)})
print(chunk_documents[0].metadata)
print(chunk_documents[0].page_content[:500])

## Create Embeddings

Embeddings are used for search. Do not inspect full vectors; inspect vector length and retrieval results.

In [ ]:
embedding_model = os.getenv("OLLAMA_EMBEDDING_MODEL", "embeddinggemma")
ollama_base_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

embeddings = OllamaEmbeddings(
    model=embedding_model,
    base_url=ollama_base_url,
)

sample_vector = embeddings.embed_query("What is prompt injection?")

print({"embedding_model": embedding_model, "vector_dimensions": len(sample_vector)})

## Build The Chroma Index

The index is generated data. Rebuilding it from chunks is safe because the source chunks are saved in JSONL.

In [ ]:
if CHROMA_PERSIST_DIR.exists():
    shutil.rmtree(CHROMA_PERSIST_DIR)

vector_store = Chroma.from_documents(
    documents=chunk_documents,
    embedding=embeddings,
    ids=chunk_ids,
    collection_name=COLLECTION_NAME,
    persist_directory=str(CHROMA_PERSIST_DIR),
)

stored = vector_store.get(include=[])

print({
    "persist_directory": str(CHROMA_PERSIST_DIR),
    "collection_name": COLLECTION_NAME,
    "stored_chunks": len(stored["ids"]),
})

## Test Retrieval Before Generation

If retrieval returns weak chunks, the final answer will also be weak. Check retrieval before adding the LLM.

In [ ]:
query = "What is prompt injection?"
results = vector_store.similarity_search_with_score(query, k=4)

print({"query": query, "results": len(results)})

for document, score in results:
    print(
        {
            "score": round(score, 4),
            "page": document.metadata.get("page"),
            "chunk_id": document.metadata.get("chunk_id"),
        }
    )
    print(document.page_content[:500])
    print("---")